# Data Cleaning: Messy Customer Dataset
**Task 3 — Data Analytics Track, Level 1**

Dataset: `messy_customer_data.csv` (included, 840 rows) — deliberately messy with missing values, duplicate rows, inconsistent gender formatting, invalid ages, and mixed date formats, to practice systematic data cleaning.


In [1]:
import pandas as pd
import numpy as np

FILENAME = 'messy_customer_data.csv'
df = pd.read_csv(FILENAME)
df.head(10)


,CustomerID,Name,Age,Gender,City,Salary,JoinDate,Email
0,1,Thomas Smith,NaN,m,Philadelphia,33381.89,04-30-2020,thomas.smith@example.com
1,2,Joseph Brown,68.0,F,Philadelphia,NaN,06-22-2023,joseph.brown@example.com
2,3,Elizabeth Jones,NaN,F,Phoenix,74859.64,13-Dec-2021,elizabeth.jones@example.com
3,4,Michael Gonzalez,75.0,Male,San Antonio,51614.40,2019-06-13,michael.gonzalez@example.com
4,5,Mary Wilson,58.0,male,Chicago,26404.45,09/10/2019,mary.wilson@example.com
5,6,Patricia Gonzalez,60.0,M,New York,124023.40,2021-10-20,patricia.gonzalez@example.com
6,7,Joseph Johnson,23.0,F,Chicago,49962.36,09-16-2021,joseph.johnson@example.com
7,8,Richard Gonzalez,45.0,male,Phoenix,62196.78,24/05/2023,richard.gonzalez@example.com
8,9,Susan Jones,33.0,m,Chicago,72069.69,2018-11-29,susan.jones@example.com
9,10,Linda Anderson,62.0,female,Philadelphia,56322.32,08/12/2022,linda.anderson@example.com


## 1. Data Quality Report (Before Cleaning)


In [2]:
def data_quality_report(data, label):
    print(f'--- {label} ---')
    print('Rows:', len(data))
    print('Duplicate rows (all columns):', data.duplicated().sum())
    print('Duplicate rows (excluding CustomerID):', data.drop(columns=['CustomerID'], errors='ignore').duplicated().sum())
    print('\nNulls per column:')
    print(data.isnull().sum())
    print('\nDtypes:')
    print(data.dtypes)
    print()

data_quality_report(df, 'BEFORE CLEANING')


--- BEFORE CLEANING ---
Rows: 840
Duplicate rows (all columns): 0
Duplicate rows (excluding CustomerID): 40

Nulls per column:
CustomerID     0
Name           0
Age           96
Gender         0
City           0
Salary        76
JoinDate       0
Email         34
dtype: int64

Dtypes:
CustomerID      int64
Name              str
Age           float64
Gender            str
City              str
Salary        float64
JoinDate          str
Email             str
dtype: object



In [3]:
# Value range anomalies check
print('Age range:', df['Age'].min(), 'to', df['Age'].max())
print('Salary range:', df['Salary'].min(), 'to', df['Salary'].max())
print('\nUnique Gender values:', df['Gender'].unique())
print('\nSample JoinDate values:', df['JoinDate'].head(10).tolist())


Age range: -5.0 to 200.0
Salary range: 8762.43 to 181992.65

Unique Gender values: <StringArray>
['m', 'F', 'Male', 'male', 'M', 'female', 'Female', 'f']
Length: 8, dtype: str

Sample JoinDate values: ['04-30-2020', '06-22-2023', '13-Dec-2021', '2019-06-13', '09/10/2019', '2021-10-20', '09-16-2021', '24/05/2023', '2018-11-29', '08/12/2022']


## 2. Missing Data Handling

**Strategy per column (documented below):**
- **Age**: median imputation. Age is numeric with a roughly symmetric distribution once invalid values are removed, so the median is a robust central estimate that won't be skewed by outliers.
- **Salary**: median imputation. Salary is subject to a right-skew (a few very high earners), so median is safer than mean to avoid overestimating typical salary.
- **Email**: leave as missing (do not impute). An email address cannot be reasonably guessed or derived — imputing a fake email would introduce false data. Rows are kept, just flagged as missing.


In [4]:
# First, fix invalid Age values (negative or unrealistically high) by treating them as missing
df.loc[(df['Age'] < 0) | (df['Age'] > 100), 'Age'] = np.nan

before_age_nulls = df['Age'].isnull().sum()
before_salary_nulls = df['Salary'].isnull().sum()

age_median = df['Age'].median()
salary_median = df['Salary'].median()

df['Age'] = df['Age'].fillna(age_median)
df['Salary'] = df['Salary'].fillna(salary_median)

print(f'Age: filled {before_age_nulls} missing/invalid values with median ({age_median:.1f})')
print(f'Salary: filled {before_salary_nulls} missing values with median ({salary_median:.2f})')
print(f'Email: left {df["Email"].isnull().sum()} missing values as-is (cannot be reasonably imputed)')


Age: filled 131 missing/invalid values with median (45.0)
Salary: filled 76 missing values with median (54747.56)
Email: left 34 missing values as-is (cannot be reasonably imputed)


## 3. Duplicate Removal


In [5]:
# Duplicates are checked excluding CustomerID, since CustomerID is a unique row identifier
# and would never match even for otherwise-identical rows
before_rows = len(df)
duplicate_mask = df.drop(columns=['CustomerID']).duplicated(keep='first')
num_duplicates = duplicate_mask.sum()

df = df[~duplicate_mask].reset_index(drop=True)

print(f'Removed {num_duplicates} duplicate rows.')
print(f'Row count: {before_rows} -> {len(df)}')


Removed 40 duplicate rows.
Row count: 840 -> 800


## 4. Standardisation

Normalising inconsistent formatting: Gender values and date formats.


In [6]:
# Standardise Gender: map all variants to 'Male' / 'Female'
gender_map = {
    'male': 'Male', 'm': 'Male', 'Male': 'Male',
    'female': 'Female', 'f': 'Female', 'Female': 'Female'
}
df['Gender'] = df['Gender'].str.strip().str.lower().map(
    {'male': 'Male', 'm': 'Male', 'female': 'Female', 'f': 'Female'}
)

print('Gender values after standardisation:', df['Gender'].unique())


Gender values after standardisation: <StringArray>
['Male', 'Female']
Length: 2, dtype: str


In [7]:
# Standardise JoinDate: parse mixed formats into a single datetime dtype
def parse_mixed_date(date_str):
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%m-%d-%Y', '%d-%b-%Y'):
        try:
            return pd.to_datetime(date_str, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT  # could not parse

df['JoinDate'] = df['JoinDate'].apply(parse_mixed_date)

print('JoinDate dtype after standardisation:', df['JoinDate'].dtype)
print('Unparseable dates remaining:', df['JoinDate'].isnull().sum())
df[['JoinDate']].head(10)


JoinDate dtype after standardisation: datetime64[us]
Unparseable dates remaining: 0


,JoinDate
0,2020-04-30
1,2023-06-22
2,2021-12-13
3,2019-06-13
4,2019-10-09
5,2021-10-20
6,2021-09-16
7,2023-05-24
8,2018-11-29
9,2022-12-08


## 5. Outlier Detection (IQR Method)


In [8]:
def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    return outliers, lower_bound, upper_bound

salary_outliers, sal_low, sal_high = detect_outliers_iqr(df['Salary'])
print(f'Salary outliers found: {len(salary_outliers)} (bounds: {sal_low:.2f} to {sal_high:.2f})')

age_outliers, age_low, age_high = detect_outliers_iqr(df['Age'])
print(f'Age outliers found: {len(age_outliers)} (bounds: {age_low:.2f} to {age_high:.2f})')


Salary outliers found: 32 (bounds: 18360.29 to 91515.68)
Age outliers found: 0 (bounds: -4.50 to 95.50)


**Decision on outliers (documented):**
- **Salary outliers**: cap (clip) rather than remove. Very high or low salaries are plausible in real data (executives vs. entry-level) — removing them would lose valid information, so they are capped to the IQR bounds instead of dropped.
- **Age outliers**: since invalid ages (negative, 150+) were already handled as missing data in Section 2, remaining statistical outliers are retained as-is — they represent plausible ages (e.g. a 75-year-old customer), not data errors.


In [9]:
# Cap Salary outliers to the IQR bounds
df['Salary'] = df['Salary'].clip(lower=sal_low, upper=sal_high)
print('Salary capped to IQR bounds. New range:', df['Salary'].min(), 'to', df['Salary'].max())


Salary capped to IQR bounds. New range: 18360.293750000008 to 91515.68375


## 6. Data Type Correction


In [10]:
df['CustomerID'] = df['CustomerID'].astype(str)
df['Age'] = df['Age'].astype(int)
df['Salary'] = df['Salary'].astype(float)
# JoinDate is already datetime from Section 4

print(df.dtypes)


CustomerID               str
Name                     str
Age                    int64
Gender                   str
City                     str
Salary               float64
JoinDate      datetime64[us]
Email                    str
dtype: object


## 7. Data Quality Report (After Cleaning)


In [11]:
data_quality_report(df, 'AFTER CLEANING')


--- AFTER CLEANING ---
Rows: 800
Duplicate rows (all columns): 0
Duplicate rows (excluding CustomerID): 0

Nulls per column:
CustomerID     0
Name           0
Age            0
Gender         0
City           0
Salary         0
JoinDate       0
Email         33
dtype: int64

Dtypes:
CustomerID               str
Name                     str
Age                    int64
Gender                   str
City                     str
Salary               float64
JoinDate      datetime64[us]
Email                    str
dtype: object



## 8. Before vs. After Summary Table


In [12]:
summary = pd.DataFrame({
    'Metric': ['Row count', 'Duplicate rows', 'Null values (Age)', 'Null values (Salary)',
               'Null values (Email)', 'Age dtype', 'Salary dtype', 'JoinDate dtype'],
    'Before': [before_rows, num_duplicates, before_age_nulls, before_salary_nulls,
               df['Email'].isnull().sum(), 'float64 (with invalid values)', 'float64', 'object (mixed strings)'],
    'After': [len(df), 0, 0, 0, df['Email'].isnull().sum(), 'int64', 'float64', 'datetime64[ns]']
})
summary


,Metric,Before,After
0,Row count,840,800
1,Duplicate rows,40,0
2,Null values (Age),131,0
3,Null values (Salary),76,0
4,Null values (Email),33,33
5,Age dtype,float64 (with invalid values),int64
6,Salary dtype,float64,float64
7,JoinDate dtype,object (mixed strings),datetime64[ns]


**Observation:** *Cleaning reduced the dataset from 840 to 800 rows by removing 40 exact duplicate rows. All 131 invalid/missing Age values and 76 missing Salary values were resolved through median imputation, bringing both columns to zero nulls. Email was intentionally left with 33 missing values since a missing email cannot be reasonably inferred. Data types were corrected across the board — Age became a proper integer, and JoinDate was converted from inconsistent string formats into a unified datetime type, making the dataset fully ready for analysis.


## 9. Save Cleaned Dataset


In [13]:
df.to_csv('cleaned_customer_data.csv', index=False)
print('Cleaned dataset saved as cleaned_customer_data.csv')
print('Final shape:', df.shape)


Cleaned dataset saved as cleaned_customer_data.csv
Final shape: (800, 8)


## Conclusion

This dataset had five main data quality issues: missing values in Age, Salary, and Email; invalid Age entries (negative numbers and ages over 100); inconsistent Gender formatting (8 different variants representing just 2 categories); mixed date formats across four different patterns; and 40 duplicate customer records.
Each issue was resolved with a documented, justified approach: invalid ages were first converted to missing, then both Age and Salary were filled using median imputation (chosen over mean to avoid distortion from outliers). Email was deliberately left missing rather than falsely imputed. Duplicates were removed based on all columns except the unique CustomerID. Gender was standardized by normalizing case and whitespace before mapping to two clean categories. Dates were parsed by attempting multiple known formats until one matched successfully. Salary outliers were capped rather than removed, since unusually high salaries (e.g. executives) are legitimate data points, not errors.
With more time or domain knowledge, I would investigate whether the missing Salary values correlate with a specific City or Age group (which could suggest a systematic data collection issue rather than random missingness), and would consult a data dictionary to confirm whether email absence indicates opted-out customers rather than a collection failure.
